<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/eu_ai_actRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy scikit-learn sentence-transformers faiss-cpu rank-bm25 pymupdf requests beautifulsoup4 lxml google-genai tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 59.0 MB/s eta 0:00:00


In [1]:
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dataclasses import dataclass

import fitz
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai

In [9]:
DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
EVAL_DIR = os.path.join(DATA_DIR, "eval")
for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "eu_ai_act.pdf") # Added this line
PDF_URL = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689"

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF from {PDF_URL} to {PDF_PATH}")
    response = requests.get(PDF_URL)
    response.raise_for_status() # Raise an exception for bad status codes
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)
    print("Download complete.")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL = "gemini-2.5-flash"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_BM25 = 20
TOP_K_DENSE = 20
TOP_K_RERANK = 5

Download complete.


In [10]:
import os
from google.colab import userdata

# Fetch the Gemini API key from the Colab secrets manager
GEMINI_API_KEY = userdata.get('GoogleAIAPI')

if not GEMINI_API_KEY:
    raise ValueError("Please set GEMINI_API_KEY in your environment or Colab secrets (named 'GoogleAIAPI') first.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [11]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    metadata: dict

def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_long_text(text, max_chars=1800, overlap=200):
    text = clean_text(text)
    if len(text) <= max_chars:
        return [text]
    parts = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        parts.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap
    return [p for p in parts if p]

In [12]:
def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    rows = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = clean_text(page.get_text("text"))
        if text:
            rows.append({
                "doc_id": "eu_ai_act",
                "page": i + 1,
                "title": "EU AI Act",
                "text": text
            })
    return pd.DataFrame(rows)


pages_df = extract_pdf_pages(PDF_PATH)
pages_df.head()

,doc_id,page,title,text
0,eu_ai_act,1,EU AI Act,REGULATION (EU) 2024/1689 OF THE EUROPEAN PARL...
1,eu_ai_act,2,EU AI Act,guaranteeing the uniform protection of overrid...
2,eu_ai_act,3,EU AI Act,(9)\nHarmonised rules applicable to the placin...
3,eu_ai_act,4,EU AI Act,rights and guarantees awarded to them by such ...
4,eu_ai_act,5,EU AI Act,(16)\nThe notion of ‘biometric categorisation’...


In [13]:
def build_chunks_from_pdf_pages(pages_df):
    chunks = []
    for _, row in pages_df.iterrows():
        parts = split_long_text(row["text"], max_chars=1800, overlap=200)
        for j, part in enumerate(parts):
            chunks.append(
                Chunk(
                    chunk_id=f"{row['doc_id']}_page_{int(row['page'])}_{j}",
                    doc_id=row["doc_id"],
                    text=part,
                    metadata={
                        "title": row["title"],
                        "page": int(row["page"]),
                        "part": j
                    }
                )
            )
    return chunks

chunks = build_chunks_from_pdf_pages(pages_df)
len(chunks), chunks[0]

(433,
 Chunk(chunk_id='eu_ai_act_page_1_0', doc_id='eu_ai_act', text='REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL\nof 13 June 2024\nlaying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, \n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and \nDirectives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)\n(Text with EEA relevance)\nTHE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,\nHaving regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,\nHaving regard to the proposal from the European Commission,\nAfter transmission of the draft legislative act to the national parliaments,\nHaving regard to the opinion of the European Economic and Social Committee (1),\nHaving regard to the opinion of the European Central Bank (2),\nHaving regard to the opinion of the Committee of the Regions (3),\nA

In [14]:
def chunks_to_dataframe(chunks):
    rows = []
    for c in chunks:
        row = {
            "chunk_id": c.chunk_id,
            "doc_id": c.doc_id,
            "text": c.text
        }
        row.update(c.metadata)
        rows.append(row)
    return pd.DataFrame(rows)

chunks_df = chunks_to_dataframe(chunks)
chunks_df.to_csv(os.path.join(PROCESSED_DIR, "chunks.csv"), index=False)
chunks_df.head()

,chunk_id,doc_id,text,title,page,part
0,eu_ai_act_page_1_0,eu_ai_act,REGULATION (EU) 2024/1689 OF THE EUROPEAN PARL...,EU AI Act,1,0
1,eu_ai_act_page_1_1,eu_ai_act,"e of law and environmental protection, to prot...",EU AI Act,1,1
2,eu_ai_act_page_1_2,eu_ai_act,obligations for operators and \nOfficial Journ...,EU AI Act,1,2
3,eu_ai_act_page_2_0,eu_ai_act,guaranteeing the uniform protection of overrid...,EU AI Act,2,0
4,eu_ai_act_page_2_1,eu_ai_act,"ty, justice, resource and energy \nefficiency,...",EU AI Act,2,1


In [15]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

chunk_texts = [c.text for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(chunk_embeddings)

print("Chunks:", len(chunks))
print("Embedding dim:", dim)
print("FAISS size:", faiss_index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Chunks: 433
Embedding dim: 384
FAISS size: 433


In [16]:
tokenized_corpus = [c.text.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [17]:
reranker = CrossEncoder(RERANK_MODEL_NAME)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [18]:
def dense_retrieve(query, top_k=10):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = faiss_index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[idx]
        results.append({"chunk": c, "score": float(score), "method": "dense"})
    return results

def bm25_retrieve(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    idxs = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in idxs:
        c = chunks[idx]
        results.append({"chunk": c, "score": float(scores[idx]), "method": "bm25"})
    return results

def reciprocal_rank_fusion(result_lists, k=60):
    fused = {}
    chunk_map = {}
    for res_list in result_lists:
        for rank, item in enumerate(res_list, start=1):
            cid = item["chunk"].chunk_id
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (k + rank)
            chunk_map[cid] = item["chunk"]
    merged = [{"chunk": chunk_map[cid], "score": score} for cid, score in fused.items()]
    merged.sort(key=lambda x: x["score"], reverse=True)
    return merged

def rerank(query, candidates, top_k=5):
    pairs = [(query, c["chunk"].text) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [{"chunk": item[0]["chunk"], "score": float(item[1])} for item in ranked[:top_k]]

In [19]:
def rewrite_query(query):
    q = query.strip()
    q = re.sub(r"\bAI Act\b", "Regulation (EU) 2024/1689", q, flags=re.I)
    q = re.sub(r"\bhigh risk\b", "high-risk AI system", q, flags=re.I)
    return q

In [20]:
def generate_with_gemini(prompt, model=GEMINI_MODEL):
    response = client.models.generate_content(
        model=model,
        contents=prompt
    )
    return response.text

In [21]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join([
        f"[{i+1}] Page {c.metadata.get('page','?')}:\n{c.text}"
        for i, c in enumerate(retrieved_chunks)
    ])
    return f"""
You are an expert EU legal assistant.
Answer only from the provided context.
If the answer is not supported, say you do not have enough evidence.
Cite the page numbers where relevant.

Question:
{query}

Context:
{context}

Answer:
""".strip()

In [22]:
def baseline_answer(query, top_k=5):
    dense_hits = dense_retrieve(query, top_k=TOP_K_DENSE)
    top_chunks = [x["chunk"] for x in dense_hits[:top_k]]
    prompt = build_prompt(query, top_chunks)
    answer = generate_with_gemini(prompt)
    return {
        "query": query,
        "rewritten_query": query,
        "retrieved": top_chunks,
        "answer": answer
    }

In [23]:
def enhanced_answer(query, top_k=5):
    q = rewrite_query(query)

    bm25_hits = bm25_retrieve(q, top_k=TOP_K_BM25)
    dense_hits = dense_retrieve(q, top_k=TOP_K_DENSE)

    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    top_for_rerank = fused[:20]

    reranked = rerank(q, top_for_rerank, top_k=top_k)
    top_chunks = [x["chunk"] for x in reranked]

    prompt = build_prompt(query, top_chunks)
    answer = generate_with_gemini(prompt)

    return {
        "query": query,
        "rewritten_query": q,
        "retrieved": top_chunks,
        "answer": answer
    }

In [24]:
test_queries = [
    "What is the purpose of the AI Act?",
    "When do the transparency rules start to apply?",
    "What are high-risk AI systems?",
    "Which AI practices are prohibited?",
    "What obligations apply to providers of GPAI models?"
]

In [25]:
for q in test_queries[:3]:
    print("=" * 120)
    print("QUESTION:", q)
    try:
        res = enhanced_answer(q)
        print("\nANSWER:\n", res["answer"])
        print("\nRETRIEVED PAGES:", [c.metadata.get("page") for c in res["retrieved"]])
    except Exception as e:
        print("Error:", e)

QUESTION: What is the purpose of the AI Act?

ANSWER:
 The purpose of the AI Act (Regulation (EU) 2024/1689) is to:
*   Improve the functioning of the internal market by laying down a uniform legal framework for the development, placing on the market, putting into service, and use of artificial intelligence systems (AI systems) in the Union, in accordance with Union values.
*   Promote the uptake of human-centric and trustworthy artificial intelligence (AI).
*   Ensure a high level of protection of health, safety, fundamental rights (including democracy, the rule of law, and environmental protection).
*   Protect against the harmful effects of AI systems in the Union.
*   Support innovation.
*   Ensure the free movement, cross-border, of AI systems.

(Page 1)

RETRIEVED PAGES: [1, 121, 118, 119, 120]
QUESTION: When do the transparency rules start to apply?

ANSWER:
 The provided context states that transparency obligations are laid down in Article 50 (Page 114). However, the specific a

In [26]:
eval_data = [
    {
        "query": "What is the purpose of the AI Act?",
        "gold_answer": "The AI Act aims to regulate AI systems according to risk and support trustworthy AI while protecting fundamental rights.",
        "gold_pages": [1, 2, 3]
    },
    {
        "query": "When do the transparency rules start to apply?",
        "gold_answer": "Transparency rules begin to apply on 2 August 2026.",
        "gold_pages": [40, 41, 42]
    },
    {
        "query": "What AI practices are prohibited?",
        "gold_answer": "Prohibited practices are those classified as unacceptable risk under the Act.",
        "gold_pages": [4, 5, 6]
    }
]

eval_df = pd.DataFrame(eval_data)
eval_df.to_csv(os.path.join(EVAL_DIR, "eval_questions.csv"), index=False)
eval_df

,query,gold_answer,gold_pages
0,What is the purpose of the AI Act?,The AI Act aims to regulate AI systems accordi...,"[1, 2, 3]"
1,When do the transparency rules start to apply?,Transparency rules begin to apply on 2 August ...,"[40, 41, 42]"
2,What AI practices are prohibited?,Prohibited practices are those classified as u...,"[4, 5, 6]"


In [27]:
def retrieve_pages(query, method="enhanced", top_k=5):
    if method == "baseline":
        hits = dense_retrieve(query, top_k=top_k)
        return [h["chunk"].metadata.get("page") for h in hits]
    q = rewrite_query(query)
    bm25_hits = bm25_retrieve(q, top_k=TOP_K_BM25)
    dense_hits = dense_retrieve(q, top_k=TOP_K_DENSE)
    fused = reciprocal_rank_fusion([bm25_hits, dense_hits])
    reranked = rerank(q, fused[:20], top_k=top_k)
    return [h["chunk"].metadata.get("page") for h in reranked]

def page_hit(retrieved_pages, gold_pages):
    return int(any(p in gold_pages for p in retrieved_pages))

In [28]:
rows = []
for _, row in eval_df.iterrows():
    q = row["query"]
    gold_pages = row["gold_pages"]

    base_ret = retrieve_pages(q, method="baseline", top_k=5)
    enh_ret = retrieve_pages(q, method="enhanced", top_k=5)

    rows.append({
        "query": q,
        "baseline_hit": page_hit(base_ret, gold_pages),
        "enhanced_hit": page_hit(enh_ret, gold_pages),
        "baseline_retrieved_pages": base_ret,
        "enhanced_retrieved_pages": enh_ret,
        "gold_pages": gold_pages
    })

retrieval_eval_df = pd.DataFrame(rows)
retrieval_eval_df

,query,baseline_hit,enhanced_hit,baseline_retrieved_pages,enhanced_retrieved_pages,gold_pages
0,What is the purpose of the AI Act?,0,1,"[129, 84, 49, 51, 26]","[1, 121, 118, 119, 120]","[1, 2, 3]"
1,When do the transparency rules start to apply?,0,0,"[114, 121, 123, 31, 44]","[44, 123, 114, 45, 39]","[40, 41, 42]"
2,What AI practices are prohibited?,0,0,"[51, 7, 41, 84, 9]","[51, 9, 8, 8, 9]","[4, 5, 6]"


In [29]:
retrieval_eval_df.to_csv(os.path.join(EVAL_DIR, "retrieval_eval.csv"), index=False)
retrieval_eval_df[["baseline_hit", "enhanced_hit"]].mean()

,0
baseline_hit,0.000000
enhanced_hit,0.333333


In [30]:
def run_and_log(method="enhanced", out_path=None):
    results = []
    for _, row in eval_df.iterrows():
        q = row["query"]
        if method == "baseline":
            res = baseline_answer(q)
        else:
            res = enhanced_answer(q)
        results.append({
            "query": q,
            "rewritten_query": res["rewritten_query"],
            "answer": res["answer"],
            "retrieved_pages": [c.metadata.get("page") for c in res["retrieved"]]
        })
    out_df = pd.DataFrame(results)
    if out_path:
        out_df.to_csv(out_path, index=False)
    return out_df

In [31]:
q = "What obligations apply to providers of GPAI models?"
try:
    out = enhanced_answer(q)
    print(out["answer"])
except Exception as e:
    print("Error:", e)

Providers of general-purpose AI models have several obligations:

1.  **Technical Documentation:** Draw up and keep up-to-date the technical documentation of the model, including its training and testing process and evaluation results. This documentation, containing information from Annex XI, must be provided upon request to the AI Office and national competent authorities (Page 84, Article 53(1)(a)).
2.  **Information and Documentation for Integrators:** Draw up, keep up-to-date, and make available information and documentation to providers of AI systems who intend to integrate the general-purpose AI model into their AI systems. This information must:
    *   Enable providers of AI systems to understand the capabilities and limitations of the model and comply with their obligations (Page 84, Article 53(1)(b)(i)).
    *   Contain, at a minimum, the elements set out in Annex XII (Page 84, Article 53(1)(b)(ii)).
3.  **Copyright Compliance Policy:** Put in place a policy to comply with Un